In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

## Langsmith tracking
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]="true"
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_SCRAPING_PROJECT")


In [ ]:
## Data Ingestion - Scraping data from website

from langchain_community.document_loaders import WebBaseLoader

# load data from website
loader = WebBaseLoader("https://en.wikipedia.org/wiki/Artificial_intelligence")
documents = loader.load()
documents


In [ ]:
# split data into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)
chunks


In [ ]:
## Ollama Embeddings
from langchain_community.embeddings import OllamaEmbeddings

embeddings = OllamaEmbeddings(model="embeddinggemma:latest")
embeddings

In [8]:
# vectorize the chunks
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore

In [10]:
# perform similarity search
query = "What is Artificial Intelligence?"
docs = vectorstore.similarity_search(query)
docs[0].page_content

'Glossary\nGlossary\nvte\nArtificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1]'

In [11]:
# perform similarity search
query = "What is Machine Learning?"
docs = vectorstore.similarity_search(query)
docs[0].page_content

'Learning\nMachine learning is the study of programs that can improve their performance on a given task automatically.[42] It has been a part of AI from the beginning.[e]'

In [ ]:
# RetrievalChain with LLM
from langchain_community.chains import RetrievalQA
from langchain_community.chat_models import OllamaChatModel
llm = OllamaChatModel(model="gemma:latest")
qa = RetrievalQA.from_chain_type(llm=llm, retriever=vectorstore.as_retriever())
qa.run("What is Artificial Intelligence?")
qa.run("What is Machine Learning?")
